In [1]:
import pandas as pd
import numpy as np
from pickle import dump

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB


In [2]:
DATASET_PATH = "fraud_detection.csv"   # update path if needed

df = pd.read_csv(DATASET_PATH)
df.head()


,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,CASH_OUT,183806.32,19391.00,0.00,382572.19,566378.51,0
1,PAYMENT,521.37,0.00,0.00,0.00,0.00,0
2,PAYMENT,3478.18,19853.00,16374.82,0.00,0.00,0
3,PAYMENT,1716.05,5769.17,4053.13,0.00,0.00,0
4,CASH_IN,253129.93,1328499.49,1581629.42,2713220.48,2460090.55,0


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28745 entries, 0 to 28744
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   type            28745 non-null  object 
 1   amount          28745 non-null  float64
 2   oldbalanceOrg   28745 non-null  float64
 3   newbalanceOrig  28745 non-null  float64
 4   oldbalanceDest  28745 non-null  float64
 5   newbalanceDest  28745 non-null  float64
 6   isFraud         28745 non-null  int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 1.5+ MB


In [4]:
df.isnull().sum()


type              0
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
dtype: int64

In [5]:
encoder = LabelEncoder()
df['type'] = encoder.fit_transform(df['type'])

# Save encoder for Django usage
dump(encoder, open("encoder.pkl", "wb"))

df.head()


,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,183806.32,19391.00,0.00,382572.19,566378.51,0
1,3,521.37,0.00,0.00,0.00,0.00,0
2,3,3478.18,19853.00,16374.82,0.00,0.00,0
3,3,1716.05,5769.17,4053.13,0.00,0.00,0
4,0,253129.93,1328499.49,1581629.42,2713220.48,2460090.55,0


In [6]:
X = df.drop(columns=['isFraud'])
y = df['isFraud']

X.shape, y.shape


((28745, 6), (28745,))

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)


In [8]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro"),
        "Recall": recall_score(y_test, y_pred, average="macro"),
        "F1-Score": f1_score(y_test, y_pred, average="macro")
    }


In [9]:
ada_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=1.0,
    random_state=0
)

ada_model.fit(X_train, y_train)

ada_results = evaluate_model(ada_model, X_test, y_test)
ada_results


C:\Users\srsri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


{'Accuracy': 0.9754739954774744,
 'Precision': np.float64(0.9687778550638829),
 'Recall': np.float64(0.9713294589305498),
 'F1-Score': np.float64(0.9700421279284501)}

In [10]:
# Save trained AdaBoost model
dump(ada_model, open("AdaBoostClassifier.pkl", "wb"))


In [11]:
lr_model = LogisticRegression(
    max_iter=1000,
    solver='lbfgs'
)

lr_model.fit(X_train, y_train)

lr_results = evaluate_model(lr_model, X_test, y_test)
lr_results


{'Accuracy': 0.9269438163158811,
 'Precision': np.float64(0.9334966390869539),
 'Recall': np.float64(0.8856936051601518),
 'F1-Score': np.float64(0.9055726740557267)}

In [12]:
# Save trained Logistic Regression model
dump(lr_model, open("LogisticRegression.pkl", "wb"))


In [13]:
# MultinomialNB requires non-negative features
X_train_nb = X_train.clip(lower=0)
X_test_nb = X_test.clip(lower=0)

mnb_model = MultinomialNB(alpha=1.0)

mnb_model.fit(X_train_nb, y_train)

mnb_results = evaluate_model(mnb_model, X_test_nb, y_test)
mnb_results


{'Accuracy': 0.6955992346495042,
 'Precision': np.float64(0.7005068056411567),
 'Recall': np.float64(0.7454587485202688),
 'F1-Score': np.float64(0.6825171371163699)}

In [14]:
# Save trained MultinomialNB model
dump(mnb_model, open("MultinomialNB.pkl", "wb"))


In [15]:
results_df = pd.DataFrame({
    "AdaBoost": ada_results,
    "Logistic Regression": lr_results,
    "Multinomial Naive Bayes": mnb_results
}).T

results_df


,Accuracy,Precision,Recall,F1-Score
AdaBoost,0.975474,0.968778,0.971329,0.970042
Logistic Regression,0.926944,0.933497,0.885694,0.905573
Multinomial Naive Bayes,0.695599,0.700507,0.745459,0.682517


In [16]:
results_df_percentage = results_df * 100
results_df_percentage.round(2)


,Accuracy,Precision,Recall,F1-Score
AdaBoost,97.55,96.88,97.13,97.00
Logistic Regression,92.69,93.35,88.57,90.56
Multinomial Naive Bayes,69.56,70.05,74.55,68.25


In [17]:
best_model = results_df['F1-Score'].idxmax()
best_model


'AdaBoost'

In [18]:
results_df_percentage.to_csv("fraud_model_comparison_results.csv")
